# 07 — Distributed sketched ARC

Simulate multiple workers with low-rank Hessian sketches and optional network latency. Measure speedup and approximate bytes transferred.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic
from cubic_reg.solvers import arc, distributed
from cubic_reg.plotting import plot_optimality_gap

%matplotlib inline

In [ ]:
p = Quadratic(n=60, condition=80.0, seed=0)
x0 = np.ones(p.dim)
r_arc = arc.minimize(p, x0=x0, M0=1.0, eps=1e-6)
r_d = distributed.minimize(p, x0=x0, n_workers=4, sketch_rank=12, eps=1e-5, max_iter=100)
print("Central ARC", r_arc.nit, r_arc.grad_norm, r_arc.time_sec)
print("Dist sketch", r_d.nit, r_d.grad_norm, r_d.time_sec, r_d.message)
plot_optimality_gap({"ARC": r_arc, "Dist-ARC": r_d}, f_star=p.f_star)
plt.show()

In [ ]:
workers = [1, 2, 4, 8, 16]
results = distributed.speedup_experiment(
    p, worker_counts=workers, sketch_rank=10, eps=1e-4, max_iter=40, network_latency_ms=1.0
)
t1 = results[0].time_sec
speedups = [t1 / r.time_sec for r in results]
bytes_proxy = []
for r in results:
    # parse bytes≈ from message if present
    msg = r.message
    b = None
    if "bytes≈" in msg:
        b = int(msg.split("bytes≈")[-1].split(";")[0])
    bytes_proxy.append(b)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(workers, speedups, "o-")
axes[0].set_xlabel("workers"); axes[0].set_ylabel("speedup vs 1"); axes[0].set_title("Simulated speedup")
axes[0].grid(True, alpha=0.3)
axes[1].plot(workers, bytes_proxy, "s-")
axes[1].set_xlabel("workers"); axes[1].set_ylabel("bytes proxy"); axes[1].set_title("Communication volume")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
for w, r, s in zip(workers, results, speedups):
    print(f"workers={w:2d} time={r.time_sec:.3f}s speedup={s:.2f} {r.message}")